# 23 · WGFMU wake-up · L1 空连 (real B1500, no DUT)

> ⚠️ **执行前必读** ⚠️
>
> 此 notebook 连真机，**严禁接器件**。验证 wake-up 流水线整体跑通。
>
> 必须先跑过：
> - [x] 20_wgfmu_iv_sweep_dryrun (PASS)
> - [x] 22_wgfmu_wakeup_dryrun (PASS)
> - [x] 21_wgfmu_iv_sweep_realdevice (PASS, baseline 电流 1µA 量级)

参数极低 (v_pgm=-1V, v_ers=+1V, 5 cycles)，即便误接器件也不会写入。

**通过标准**：
- 无 dll 错, `complete == total`
- 5 个 read 窗都有数据
- `|i_read_mean|` 量级跟 21 一致 (RSU+电缆 baseline ~1µA)

测试人：**yhzang**


In [1]:
import sys, os
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt

from fefetlab.measurements.wgfmu import (
    ensure_wgfmu_dll_path,
    autodetect_visa_addr,
    autodetect_wgfmu_chan,
    RealWgfmuBackend,
    WakeupStage,
    WakeupReadout,
    WgfmuWakeupConfig,
    WgfmuWakeupRunner,
)

print("Python:", sys.version.split()[0])
print("project root:", ROOT)


Python: 3.13.5
project root: D:\test\B1500


## 1. 自动找 dll / VISA / 通道


In [2]:
dll = ensure_wgfmu_dll_path()
print(f"✅ wgfmu.dll: {dll}")

VISA_ADDR = autodetect_visa_addr("B1500")
print(f"✅ B1500 VISA addr: {VISA_ADDR}")

backend = RealWgfmuBackend()
backend.load()
backend.open_session(VISA_ADDR)
backend.set_timeout(60.0)
channel_ids = backend.get_channel_ids()
print(f"channels: {channel_ids}")

CHAN_ID = autodetect_wgfmu_chan(backend, prefer=201 if 201 in channel_ids else None)
print(f"✅ using CHAN_ID = {CHAN_ID}")

backend.close_session()


✅ wgfmu.dll: C:\Windows\System32\wgfmu.dll
✅ B1500 VISA addr: GPIB1::17::INSTR
channels: [201, 202, 301, 302]
✅ using CHAN_ID = 201


0

## 2. 极小 wake-up: 1 stage × 5 cycles, ±1V, 1MA 量程


In [ ]:
stages = [
    WakeupStage(
        n_cycles=5, v_pgm=-1.0, v_ers=+1.0,
        t_pgm_s=10e-6, t_ers_s=10e-6,
        rise_fall_s=1e-6, inter_pulse_s=2e-6,
        label="aircheck_1V",
    ),
]
readout = WakeupReadout(
    v_read=-0.3, t_read_s=5e-6, rise_fall_s=1e-6,
    measure_points=10, measure_average_s=200e-9,
)
cfg = WgfmuWakeupConfig(
    label="L1_aircheck_wakeup",
    chan_id=CHAN_ID,
    v_init=0.0, v_base=0.0,
    operation_mode="FASTIV",
    measure_mode="CURRENT",
    measure_current_range="1MA",   # 改: 容纳 RSU baseline
    treat_warning_as_error=False,
    timeout_s=30.0,
    notes="L1 空连, 5 cycle, ±1V 安全幅值",
)
print(f"stages: {len(stages)}, total cycles: 5, expected segments: 15")


In [ ]:
runner = WgfmuWakeupRunner(backend=backend)
result = runner.run(resource=VISA_ADDR, stages=stages, readout=readout, cfg=cfg)
print(f"✅ run() returned")
print(f"complete / total : {result.complete} / {result.total}")
print(f"issues           : {result.issues}")


## 3. QC + cycles


In [ ]:
print("=== qc_df ===")
print(result.qc_df.to_string(index=False))
print()
print("=== cycles_df ===")
cols = ["cycle_idx", "stage_label", "v_pgm", "v_ers", "v_read",
        "i_read_mean", "i_read_std", "n_samples"]
print(result.cycles_df[cols].to_string(index=False))

err = result.meta.get("error_summary", "")
wrn = result.meta.get("warning_summary", "")
print("\nerror_summary  :", err if err else "(empty)")
print("warning_summary:", wrn if wrn else "(empty)")


## 4. 波形 + per-cycle 电流


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6))

t_wf, v_wf = result.plan.waveform_samples(dt_s=2e-7)
axes[0].plot(t_wf*1e6, v_wf, color="#1f77b4", lw=0.9)
axes[0].set_xlabel("time (us)")
axes[0].set_ylabel("voltage (V)")
axes[0].set_title(f"L1 aircheck · wake-up waveform (5 cycles, ±1V) · CH{CHAN_ID}")
axes[0].grid(alpha=0.3)
axes[0].axhline(0, color="gray", lw=0.5)
for seg in result.plan.segments:
    if seg["measure_points"] > 0:
        axes[0].axvspan(seg["t_high_start_s"]*1e6, seg["t_high_end_s"]*1e6,
                        alpha=0.25, color="green")

cyc = result.cycles_df
axes[1].errorbar(cyc["cycle_idx"], cyc["i_read_mean"]*1e9,
                 yerr=cyc["i_read_std"]*1e9, fmt="o-", color="#d62728")
axes[1].set_xlabel("cycle index")
axes[1].set_ylabel("i_read_mean (nA)")
axes[1].set_title("per-cycle readout current (open-circuit baseline)")
axes[1].grid(alpha=0.3)
axes[1].axhline(0, color="gray", lw=0.5)
plt.tight_layout()
plt.show()

i_abs_max = cyc["i_read_mean"].abs().max()
print(f"\n|i_read_mean|_max = {i_abs_max*1e6:.3f} uA")
if i_abs_max < 10e-6:
    print(f"✅ L1 wake-up PASS (baseline {i_abs_max*1e6:.2f} uA, 跟 21 应同量级)")
else:
    print(f"⚠️  baseline 偏高 ({i_abs_max*1e6:.2f} uA), 检查探针/接地")


In [ ]:
print("output paths:")
for k, v in result.paths.items():
    print(f"  {k:20s} -> {v}")


---

## 通过标准回顾

- [ ] cell 1-2: dll/VISA/通道 OK
- [ ] cell 3-4: `complete == total`, cycles_df 5 行齐
- [ ] cell 5: 波形看到 5 cycle 完整结构
- [ ] cell 6: `|i_read|` 跟 21 IV 同量级 (RSU baseline)

**L1 全过 → yhzang 让 Hermes 写 24/25 真器件 notebook, 那时才接器件**
